In [ ]:
# crossval_ridge_lasso_logreg_breast_cancer.py
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# 1) Data
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# 2) Pipeline: Standardize -> LogisticRegression
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, solver="saga"))  # saga supports l1 & l2
])

# 3) Hyperparameter grid: search L1 vs L2 and C on log scale
param_grid = {
    "model__penalty": ["l1", "l2"],
    "model__C": np.logspace(-3, 3, 13),  # 0.001 ... 1000
}

# 4) Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 5) GridSearchCV (optimize by ROC AUC; you can change to 'accuracy' if preferred)
gs = GridSearchCV(
    pipe,
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,  # refit on the whole train set using best params
    verbose=0
)
gs.fit(X_train, y_train)

print("Best CV score (ROC AUC):", gs.best_score_)
print("Best params:", gs.best_params_)

# 6) Evaluate on hold-out test set
y_pred = gs.predict(X_test)
y_proba = gs.predict_proba(X_test)[:, 1]
print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification report (test):\n", classification_report(y_test, y_pred))

# 7) Inspect sparsity vs shrinkage
best_model = gs.best_estimator_.named_steps["model"]
coefs = best_model.coef_.ravel()
nonzero = np.count_nonzero(coefs)

print("\nPenalty used:", best_model.penalty)
print("C used:", best_model.C)
print(f"Number of non-zero coefficients: {nonzero} / {coefs.size}")

# Optional: compare directly-trained best L1 vs best L2 models (same CV split)
# to see sparsity differences side-by-side.
param_grid_split = {
    "l1": {"model__penalty": ["l1"], "model__C": np.logspace(-3, 3, 13)},
    "l2": {"model__penalty": ["l2"], "model__C": np.logspace(-3, 3, 13)},
}
results = {}
for pen, grid in param_grid_split.items():
    gs_pen = GridSearchCV(pipe, grid, scoring="roc_auc", cv=cv, n_jobs=-1, refit=True)
    gs_pen.fit(X_train, y_train)
    mdl = gs_pen.best_estimator_.named_steps["model"]
    coefs = mdl.coef_.ravel()
    results[pen] = {
        "best_C": mdl.C,
        "cv_auc": gs_pen.best_score_,
        "test_auc": roc_auc_score(y_test, gs_pen.predict_proba(X_test)[:, 1]),
        "test_acc": accuracy_score(y_test, gs_pen.predict(X_test)),
        "non_zero": int(np.count_nonzero(coefs)),
        "total_features": coefs.size
    }

print("\nSide-by-side (best within each penalty):")
for pen in ("l1", "l2"):
    r = results[pen]
    print(
        f"{pen.upper()}: C={r['best_C']}, "
        f"CV AUC={r['cv_auc']:.4f}, Test AUC={r['test_auc']:.4f}, "
        f"Test Acc={r['test_acc']:.4f}, Non-zero={r['non_zero']}/{r['total_features']}"
    )


Best CV score (ROC AUC): 0.9958720330237357
Best params: {'model__C': np.float64(1.0), 'model__penalty': 'l2'}

Test Accuracy: 0.9824561403508771
Test ROC AUC: 0.9953703703703703

Classification report (test):
               precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


Penalty used: l2
C used: 1.0
Number of non-zero coefficients: 30 / 30

Side-by-side (best within each penalty):
L1: C=3.1622776601683795, CV AUC=0.9949, Test AUC=0.9964, Test Acc=0.9912, Non-zero=21/30
L2: C=1.0, CV AUC=0.9959, Test AUC=0.9954, Test Acc=0.9825, Non-zero=30/30
